In [1]:
#Here is a basic implementation using the MarianMT model for translation (for example, English to French):

In [2]:
!pip install transformers datasets

In [3]:
from transformers import MarianMTModel, MarianTokenizer


# Load the pre-trained model and tokenizer for English to French translation
model_weights = 'Helsinki-NLP/opus-mt-en-fr'
tokenizer = MarianTokenizer.from_pretrained(model_weights)
model = MarianMTModel.from_pretrained(model_weights)


2025-06-11 19:06:09.265523: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ImportError: 
MarianTokenizer requires the SentencePiece library but it was not found in your environment. Checkout the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


In [ ]:
# Datasets have this format...
data = [
    {"translation": {"en": "Hello, how are you?", "fr": "Bonjour, comment ça va?"}},
    {"translation": {"en": "I love programming.", "fr": "J'aime programmer."}},
    {"translation": {"en": "This is a machine translation example.", "fr": "Ceci est un exemple de traduction automatique."}},
]



In [ ]:
# Tokenize the input sentences (English in this case)
def encode_sentences(sentences, tokenizer, max_length=40):
    tokens = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    return tokens



In [ ]:
# Translate function
def translate(sentences, model, tokenizer):
    # Tokenize input
    tokenized_input = encode_sentences(sentences, tokenizer)

    # Perform translation
    translated_tokens = model.generate(**tokenized_input)

    # Decode the translation into readable text
    translated_sentences = [tokenizer.decode(t, skip_special_tokens=True) for t in translated_tokens]

    return translated_sentences



In [ ]:
# Test example sentences for translation (English to French)
source_sentences = [
    "Hello, how are you?",
    "I love programming.",
    "This is a machine translation example."
]

# Translate the sentences
translated_sentences = translate(source_sentences, model, tokenizer)

# Print results
for src, tgt in zip(source_sentences, translated_sentences):
    print(f"Source: {src} -> Translated: {tgt}")


Source: Hello, how are you? -> Translated: Bonjour, comment allez-vous?
Source: I love programming. -> Translated: J'adore la programmation.
Source: This is a machine translation example. -> Translated: C'est un exemple de traduction automatique.


# **Large Dataset**

In [4]:
!pip install -U datasets fsspec

  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.0
    Uninstalling fsspec-2024.6.0:
      Successfully uninstalled fsspec-2024.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.6.0 requires fsspec==2024.6.0, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
from datasets import load_dataset



# Load a dataset for English-French translation (e.g., 'opus_books')
dataset = load_dataset("opus_books", "en-fr")


# Use select() to get the first 10 items from the dataset
subset = dataset['train'].select(range(10))

# Prepare English and French sentences
en_sentences = [item['translation']['en'] for item in subset]
fr_sentences = [item['translation']['fr'] for item in subset]

# Print the sentences to verify
print("English Sentences:", en_sentences)
print("French Sentences:", fr_sentences)


English Sentences: ['The Wanderer', 'Alain-Fournier', 'First Part', 'I', 'THE BOARDER', 'He arrived at our home on a Sunday of November, 189-.', "I still say 'our home,' although the house no longer belongs to us.", 'We left that part of the country nearly fifteen years ago and shall certainly never go back to it.', "We were living in the building of the Higher Elementary Classes at Sainte-Agathe's School.", "My father, whom I used to call M. Seurel as did other pupils, was head of the Middle School and also of the Higher Elementary classes where pupils worked for the preliminary teacher's examination."]
French Sentences: ['Le grand Meaulnes', 'Alain-Fournier', 'PREMIÈRE PARTIE', 'CHAPITRE PREMIER', 'LE PENSIONNAIRE', 'Il arriva chez nous un dimanche de novembre 189-…', 'Je continue à dire « chez nous », bien que la maison ne nous appartienne plus.', 'Nous avons quitté le pays depuis bientôt quinze ans et nous n’y reviendrons certainement jamais.', 'Nous habitions les bâtiments du Cour

In [ ]:
print(dataset['train'][0])  # This will print the structure of one item in the dataset


{'id': '0', 'translation': {'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}}


In [ ]:

# Translate English to French
translated_fr_sentences = translate(en_sentences, model, tokenizer)

# Print some sample translations
for en, fr, pred_fr in zip(en_sentences, fr_sentences, translated_fr_sentences):
    print(f"Source: {en} -> Target: {fr} -> Predicted: {pred_fr}")


Source: The Wanderer -> Target: Le grand Meaulnes -> Predicted: Le Wanderer
Source: Alain-Fournier -> Target: Alain-Fournier -> Predicted: Alain-Fournier
Source: First Part -> Target: PREMIÈRE PARTIE -> Predicted: Première partie
Source: I -> Target: CHAPITRE PREMIER -> Predicted: Annexe I
Source: THE BOARDER -> Target: LE PENSIONNAIRE -> Predicted: LE CONSEIL D'ADMINISTRATION
Source: He arrived at our home on a Sunday of November, 189-. -> Target: Il arriva chez nous un dimanche de novembre 189-… -> Predicted: Il est arrivé à notre maison un dimanche de Novembre, 189-.
Source: I still say 'our home,' although the house no longer belongs to us. -> Target: Je continue à dire « chez nous », bien que la maison ne nous appartienne plus. -> Predicted: Je dis toujours "notre maison", bien que la maison ne nous appartienne plus.
Source: We left that part of the country nearly fifteen years ago and shall certainly never go back to it. -> Target: Nous avons quitté le pays depuis bientôt quinze 



---



In [ ]:

# Select a small subset of the data for demonstration
#train_data = dataset['train'].select(range(100))  # Use a small subset for training


In [ ]:
# Split the dataset into train and eval datasets
split_dataset = dataset['train'].select(range(100)).train_test_split(test_size=0.1)
train_data = split_dataset['train']
eval_data = split_dataset['test']


In [ ]:
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Load the pre-trained MarianMT model and tokenizer
model_weights = 'Helsinki-NLP/opus-mt-en-fr'
tokenizer = MarianTokenizer.from_pretrained(model_weights)
model = MarianMTModel.from_pretrained(model_weights)


/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# Tokenize the dataset
def preprocess_function(examples):
    inputs = [ex['en'] for ex in examples['translation']]
    targets = [ex['fr'] for ex in examples['translation']]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs

# Tokenize dataset
tokenized_train_data = train_data.map(preprocess_function, batched=True)
tokenized_eval_data = eval_data.map(preprocess_function, batched=True)


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

## Seq2Seq is used when ever the Decoder architecture is used

In [ ]:
num_epochs = 3 # in production we need at least 100 for a good PoC

In [ ]:
#For old version of transformer package -- For colab default settings, use this code
training_args = Seq2SeqTrainingArguments(
    output_dir='./results',
    do_eval=True,                     # explicitly enable evaluation
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=500,                   # optional: how often to save
    eval_steps=500                    # optional: how often to evaluate
)

In [ ]:
# Define training arguments
'''
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # Enable evaluation at the end of every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True
)
'''

In [ ]:
# Data collator for padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)



In [ ]:
# Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_data,  # Training dataset
    eval_dataset=tokenized_eval_data,    # Evaluation dataset
    data_collator=data_collator,
    tokenizer=tokenizer
)


In [ ]:

# Fine-tune the model
trainer.train()


Epoch,Training Loss,Validation Loss
1,No log,1.957163
2,No log,1.933650
3,No log,1.924908


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[59513]], 'forced_eos_token_id': 0}


TrainOutput(global_step=18, training_loss=1.966268539428711, metrics={'train_runtime': 300.5086, 'train_samples_per_second': 0.898, 'train_steps_per_second': 0.06, 'total_flos': 6532320854016.0, 'train_loss': 1.966268539428711, 'epoch': 3.0})

In [ ]:
# Save the fine-tuned model and tokenizer
model.save_pretrained("./my_custom_marian_mt")
tokenizer.save_pretrained("./my_custom_marian_mt")


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[59513]], 'forced_eos_token_id': 0}


('./my_custom_marian_mt/tokenizer_config.json',
 './my_custom_marian_mt/special_tokens_map.json',
 './my_custom_marian_mt/vocab.json',
 './my_custom_marian_mt/source.spm',
 './my_custom_marian_mt/target.spm',
 './my_custom_marian_mt/added_tokens.json')

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Load your fine-tuned model and tokenizer
model = MarianMTModel.from_pretrained("./my_custom_marian_mt")
tokenizer = MarianTokenizer.from_pretrained("./my_custom_marian_mt")

# Function to translate a sentence from English to French
def translate(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    translated_tokens = model.generate(**inputs)
    return tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

# User input loop for translation
while True:
    # Get user input
    english_sentence = input("Enter an English sentence (or 'exit' to quit): ")

    # Check if the user wants to exit
    if english_sentence.lower() == 'exit':
        print("Exiting the translation system.")
        break

    # Translate the sentence
    french_translation = translate(english_sentence)

    # Output the translated sentence
    print(f"French Translation: {french_translation}\n")


/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Enter an English sentence (or 'exit' to quit): Hello. My name is Prashant Nair and I will be your instructor for NLP course
French Translation: Bonjour, je m'appelle Prashant Nair et je serai votre instructeur pour le cours NLP.

Enter an English sentence (or 'exit' to quit): exit
Exiting the translation system.


In [ ]:
!zip -r mymodel.zip my_custom_marian_mt/


  adding: my_custom_marian_mt/ (stored 0%)
  adding: my_custom_marian_mt/model.safetensors (deflated 7%)
  adding: my_custom_marian_mt/config.json (deflated 61%)
  adding: my_custom_marian_mt/source.spm (deflated 49%)
  adding: my_custom_marian_mt/generation_config.json (deflated 43%)
  adding: my_custom_marian_mt/target.spm (deflated 50%)
  adding: my_custom_marian_mt/special_tokens_map.json (deflated 35%)
  adding: my_custom_marian_mt/vocab.json (deflated 70%)
  adding: my_custom_marian_mt/tokenizer_config.json (deflated 68%)
